# Stick-figure image batch (SD1.5, T4)
Runtime > Change runtime type > T4 GPU, then Run all. Images go to Google Drive/video_images. Re-running skips finished files.

In [ ]:
!pip -q install diffusers transformers accelerate safetensors
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch, os, json
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
from PIL import Image
pipe = StableDiffusionPipeline.from_pretrained('stable-diffusion-v1-5/stable-diffusion-v1-5', torch_dtype=torch.float16, safety_checker=None).to('cuda')
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe.enable_attention_slicing()
OUT='/content/drive/MyDrive/video_images'; os.makedirs(OUT, exist_ok=True)
STYLE='flat 2D hand-drawn cartoon, Pip, a simple white round-headed stick figure with dot eyes, teal t-shirt, black shoes, thick black ink outlines, plain cream background, ochre ground, flat colors, minimal, '
NEG='photo, realistic, 3d, shading, gradient, detailed, painting, text, watermark, signature, blurry, multiple characters, extra limbs'

In [ ]:
# Prompts: GitHub raw prompts.txt (lines "beat<TAB>prompt"), else Drive copy, else 5 style tests.
# MODE='test' runs only the 5 style tests; set MODE='full' for all prompts.
MODE='test'
import urllib.request
RAW='https://raw.githubusercontent.com/omoustarchid/video-images/main/prompts.txt'
TESTS=['a stick figure sitting at a desk looking worried, clock on the wall',
 'a stick figure running away from a giant question mark',
 'two stick figures talking at a table with coffee cups',
 'a stick figure lying in bed awake at night, moon in the window',
 'a stick figure standing on a cliff edge looking at the sunrise']
lines=None
if MODE=='full':
    try: lines=urllib.request.urlopen(RAW,timeout=30).read().decode().splitlines()
    except Exception as e: print('GitHub prompts failed:',e)
    if not lines and os.path.exists(OUT+'/prompts.txt'): lines=open(OUT+'/prompts.txt').read().splitlines()
items=[]
if lines:
    for k,l in enumerate(l for l in lines if l.strip()):
        if '\t' in l: b,p=l.split('\t',1); items.append((int(b),p.strip()))
        else: items.append((k,l.strip()))
else:
    items=list(enumerate(TESTS))
print(len(items),'prompts')

In [ ]:
import time
W,H=768,432; t0=time.time(); n=0
for b,p in items:
    f=f'{OUT}/img_{b:04d}.png'
    if os.path.exists(f): continue
    g=torch.Generator('cuda').manual_seed(1000+b)
    im=pipe(STYLE+p, negative_prompt=NEG, width=W, height=H, num_inference_steps=22, guidance_scale=7.5, generator=g).images[0]
    im.resize((1280,720), Image.LANCZOS).save(f); n+=1
    if n%25==0: print(b, round(time.time()-t0), 's', flush=True)
print('done', n, 'images in', round(time.time()-t0), 's')